# distilgpt2 LoRA Fine-tuning

Run this notebook on **Google Colab** (Runtime → Change runtime type → T4 GPU)  
or on **Kaggle** (Settings → Accelerator → GPU T4).

When finished, download `work/distilgpt2_finetuned/` and place it in the `work/`
directory of your local repository.

## Training data
- `data/training/*.txt` — scraped multilingual corpus (~138K deduplicated lines × 10 languages = ~1.38M lines)
- `data/open-dev/` — open-dev contexts + answer chars concatenated (~100K lines)
- **Total: ~1.48M lines**

## Estimated time on a T4 GPU
| Setting | Time |
|---|---|
| Full corpus + open-dev, 3 epochs, max_length=128, batch=64 | ~2–3 hours |
| Quick test (`LIMIT = 5000`), 1 epoch | ~5 minutes |

In [ ]:
# ── 1. Clone the repository ────────────────────────────────────────────────
# Replace the URL with your own fork if needed.
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

REPO_URL = "https://github.com/sarahmakki12/intro-to-nlp-project.git"
REPO_DIR = "intro-to-nlp-project"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}

In [ ]:
# ── 2. Install dependencies ────────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 3. Verify GPU and data ─────────────────────────────────────────────────
import glob
import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

txt_files = sorted(glob.glob("data/training/*.txt"))
print(f"\nCorpus files found ({len(txt_files)}):")
total = 0
for f in txt_files:
    n = sum(1 for _ in open(f, encoding="utf-8"))
    total += n
    print(f"  {os.path.basename(f)}: {n:,} lines")
print(f"  corpus total: {total:,} lines")

open_dev_input = "data/open-dev/input.txt"
open_dev_answer = "data/open-dev/answer.txt"
if os.path.exists(open_dev_input):
    n = sum(1 for _ in open(open_dev_input, encoding="utf-8"))
    print(f"\nopen-dev: {n:,} lines")
else:
    print("\nWARNING: data/open-dev/input.txt not found")

In [ ]:
# ── 4. Train ───────────────────────────────────────────────────────────────
# LIMIT        : max lines per corpus file  (None = full ~138K/file)
# OPEN_DEV_LIMIT: max lines from open-dev   (None = all ~99K; 0 = skip open-dev)
LIMIT         = None
OPEN_DEV_LIMIT = None

limit_flag         = f"--limit {LIMIT}"               if LIMIT          else ""
open_dev_limit_flag = f"--open_dev_limit {OPEN_DEV_LIMIT}" if OPEN_DEV_LIMIT else ""

!python src/models/llm_train.py \
    --work_dir work \
    --data_dir data/training \
    --open_dev_dir data/open-dev \
    --epochs 3 \
    --batch_size 64 \
    --max_length 128 \
    --lora_r 16 \
    {limit_flag} {open_dev_limit_flag}

In [ ]:
# ── 5. Verify the saved model ──────────────────────────────────────────────
import os

save_dir = "work/distilgpt2_finetuned"
files = os.listdir(save_dir)
print(f"Files in {save_dir}:", files)

total_mb = sum(
    os.path.getsize(os.path.join(save_dir, f)) for f in files
) / (1024 ** 2)
print(f"Total size: {total_mb:.0f} MB")

# Quick sanity-check: load the model and predict one example
import sys
sys.path.insert(0, "src")
from models.llm import LLMCharModel

model = LLMCharModel.load("work")
test_ctx = "Roger, Houston. We have a go for "
pred = model.predict(test_ctx)
print(f"\nContext : {repr(test_ctx)}")
print(f"Top-3   : {repr(pred)}")

In [ ]:
# ── 6. Download the model (Google Colab only) ──────────────────────────────
# Skip this cell on Kaggle — use the output panel to download the directory.
try:
    from google.colab import files
    import shutil
    shutil.make_archive("distilgpt2_finetuned", "zip", "work", "distilgpt2_finetuned")
    files.download("distilgpt2_finetuned.zip")
    print("Download started.")
    print("Unzip into your local work/ directory:")
    print("  unzip distilgpt2_finetuned.zip -d work/")
except ImportError:
    print("Not running in Colab — download work/distilgpt2_finetuned/ manually.")